# 07 — Development comparison and fail-closed selection

This is the only model-selection notebook allowed to open development
truth. Threshold values come from one late-calibration half. For every
declared portfolio, the operating quantile is then chosen without labels on
the other half using the same exact Poisson upper workload bound used by the
final gate. Development labels compare only those preselected operating
points. A threshold is selectable only when at least five calibration block
maxima are expected beyond its quantile. Diagnostic topology and multivariate
ablations remain visible but
cannot become deployable unless the policy explicitly marks them eligible.

Holdout remains physically sealed. A useful diagnostic result is not turned
into a selected model when it fails workload, recall, or score-availability
requirements.

## 1. Setup and evaluation boundary


In [ ]:
from pathlib import Path
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def find_repository(start=Path.cwd()):
    """Find the checked-out repository when Jupyter starts in any subfolder."""
    override = os.getenv("TELCO_PROJECT_ROOT")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        start = start.resolve()
        candidates = [start, *start.parents]
        if "google.colab" in sys.modules:
            candidates += [
                Path("/content/drive/MyDrive/anomaly_detection"),
                Path("/content/drive/MyDrive/telco-anomaly-detection"),
            ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from the cloned repository, or set TELCO_PROJECT_ROOT."
    )


PROJECT_ROOT = find_repository()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import json
import math

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

from telco_anomaly.detectors import (
    alert_grid_from_score_file,
    duration_to_observations,
    partition_exposure,
)
from telco_anomaly.evaluation import (
    evaluate_cases, form_cases, poisson_rate_interval,
)
from telco_anomaly.selection import select_development_candidate
from telco_anomaly.io import (
    file_sha256,
    immutable_output_directory,
    read_json,
    require_same,
    resolve_data_root,
    write_json,
)

DATA_ROOT = resolve_data_root()
CORE_RUN_ID = os.getenv(
    "TELCO_CORE_RUN_ID", os.getenv("PON_CORE_RUN_ID", "synthetic_pon_core_v2")
)
MODEL_RUN_ID = os.getenv(
    "TELCO_MODEL_RUN_ID", os.getenv("PON_MODEL_RUN_ID", "synthetic_pon_models_v7")
)
TRUTH_RUN_ID = os.getenv("PON_TRUTH_RUN_ID", "synthetic_pon_truth_v3")
SELECTION_RUN_ID = os.getenv("PON_SELECTION_RUN_ID", "synthetic_pon_selection_v7")

RUN_ROOT = DATA_ROOT / "core" / "synthetic_pon" / CORE_RUN_ID
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
MODEL_ROOT = DATA_ROOT / "models" / "synthetic_pon" / MODEL_RUN_ID
TRUTH_ROOT = DATA_ROOT / "evaluation" / "synthetic_pon" / TRUTH_RUN_ID
DEV_TRUTH = TRUTH_ROOT / "development"
HOLDOUT_TRUTH = TRUTH_ROOT / "holdout_locked"
OUTPUT_ROOT = DATA_ROOT / "selection" / "synthetic_pon" / SELECTION_RUN_ID

core_manifest = read_json(CORE_ROOT / "manifest.json")
model_manifest = read_json(MODEL_ROOT / "model_manifest.json")
resolved_policy = read_json(MODEL_ROOT / "resolved_policy.json")
truth_manifest = read_json(TRUTH_ROOT / "truth_manifest.json")
require_same(model_manifest, core_fingerprint=core_manifest["fingerprint"])
require_same(
    truth_manifest, model_core_fingerprint=core_manifest["fingerprint"]
)
if file_sha256(PROJECT_ROOT / "src" / "telco_anomaly" / "detectors.py") != model_manifest["detectors_module_sha256"]:
    raise ValueError("Detector code changed after the model was fitted")
EVALUATION_MODULE_SHA256 = file_sha256(
    PROJECT_ROOT / "src" / "telco_anomaly" / "evaluation.py"
)
SELECTION_MODULE_SHA256 = file_sha256(
    PROJECT_ROOT / "src" / "telco_anomaly" / "selection.py"
)
if file_sha256(MODEL_ROOT / "resolved_policy.json") != model_manifest["resolved_policy_sha256"]:
    raise ValueError("The frozen model policy no longer matches its manifest")
POLICY = resolved_policy["alert_policy"]
score_path = MODEL_ROOT / model_manifest["score_files"]["development"]
threshold_score_path = (
    MODEL_ROOT / model_manifest["score_files"]["calibration_threshold"]
)
verification_score_path = (
    MODEL_ROOT / model_manifest["score_files"]["calibration_verification"]
)
thresholds = pd.read_parquet(MODEL_ROOT / "calibration_thresholds.parquet")
topology_path = CORE_ROOT / "topology_memberships.parquet"
topology = pd.read_parquet(topology_path) if topology_path.exists() else pd.DataFrame()

if not DEV_TRUTH.is_dir():
    raise FileNotFoundError("Run Notebook 03 to create development truth")
for name in ("fault_events.parquet", "fault_entity_intervals.parquet"):
    relative = f"development/{name}"
    if file_sha256(DEV_TRUTH / name) != truth_manifest["file_sha256"].get(relative):
        raise ValueError(f"Development truth file changed: {relative}")
events = pd.read_parquet(DEV_TRUTH / "fault_events.parquet")
intervals = pd.read_parquet(DEV_TRUTH / "fault_entity_intervals.parquet")

display(pd.Series({
    "scores": str(score_path),
    "truth_opened": "development only",
    "holdout_opened": False,
    "development_faults": events["fault_id"].nunique(),
}, name="value").to_frame())


## 2. Candidate portfolios and operational persistence


In [ ]:
CADENCE_SECONDS = float(model_manifest["cadence_seconds"])
aliases = {
    "rapid_self": "rapid_residual",
    "persistent_drift": "drift_cusum",
}

declared = {
    **POLICY["portfolios"],
    **POLICY.get("diagnostic_portfolios", {}),
}
available = set(model_manifest["channels"])
portfolios = {}
for name, declared_channels in declared.items():
    channels = [aliases.get(channel, channel) for channel in declared_channels]
    if set(channels) <= available:
        portfolios[name] = channels

eligible_portfolios = set(POLICY["selection"]["eligible_portfolios"])
selection_eligible = {
    name: name in eligible_portfolios for name in portfolios
}

channel_policy = {
    aliases.get(name, name): values
    for name, values in POLICY["channels"].items()
}
missing_persistence = available - set(channel_policy)
if missing_persistence:
    raise ValueError(
        f"Missing persistence policy for {sorted(missing_persistence)}"
    )
persistence_seconds = {
    channel: int(channel_policy[channel]["persistence_seconds"])
    for channel in available
}
persistence = {
    channel: duration_to_observations(seconds, CADENCE_SECONDS)
    for channel, seconds in persistence_seconds.items()
}
RECOVERY_OBSERVATIONS = duration_to_observations(
    POLICY["recovery"]["duration_seconds"], CADENCE_SECONDS
)
RECOVERY_THRESHOLD_FRACTION = float(
    POLICY["recovery"]["threshold_fraction"]
)

display(pd.DataFrame([
    {
        "portfolio": name,
        "channels": ", ".join(channels),
        "selection_eligible": selection_eligible[name],
    }
    for name, channels in portfolios.items()
]))

## 3. Build alerts once per channel

Thresholds are calibration-frozen. Development labels are used only after the
score-to-alert transformation, to compare label-free portfolio operating points.


In [ ]:
alert_grid = alert_grid_from_score_file(
    score_path,
    thresholds,
    persistence=persistence,
    recovery_consecutive=RECOVERY_OBSERVATIONS,
    cadence_seconds=CADENCE_SECONDS,
    recovery_threshold_fraction=RECOVERY_THRESHOLD_FRACTION,
)
verification_grid = alert_grid_from_score_file(
    verification_score_path,
    thresholds,
    persistence=persistence,
    recovery_consecutive=RECOVERY_OBSERVATIONS,
    cadence_seconds=CADENCE_SECONDS,
    recovery_threshold_fraction=RECOVERY_THRESHOLD_FRACTION,
)

exposure = partition_exposure(score_path, "entity_day", CADENCE_SECONDS)
verification_exposure = partition_exposure(
    verification_score_path, "entity_day", CADENCE_SECONDS
)


def score_missingness(path, channels):
    source = str(path).replace("'", "''")
    result = {}
    with duckdb.connect() as connection:
        for channel in channels:
            quoted = '"' + channel.replace('"', '""') + '"'
            global_rate, entity_p95 = connection.execute(f"""
                WITH entity_rates AS (
                    SELECT CAST(entity_id AS VARCHAR) AS entity_id,
                           avg(CASE WHEN {quoted} IS NULL THEN 1.0 ELSE 0.0 END)
                               AS missing_rate
                    FROM read_parquet('{source}')
                    GROUP BY entity_id
                )
                SELECT avg(missing_rate), quantile_cont(missing_rate, 0.95)
                FROM entity_rates
            """).fetchone()
            result[channel] = {
                "global": float(global_rate or 0.0),
                "entity_p95": float(entity_p95 or 0.0),
            }
    return result


def threshold_exceedance_audit(path, threshold_table):
    source = str(path).replace("'", "''")
    rows = []
    with duckdb.connect() as connection:
        for channel, group in threshold_table.groupby("model_id", sort=False):
            group = group.sort_values("threshold_quantile")
            quoted = '"' + str(channel).replace('"', '""') + '"'
            expressions = [f"count({quoted})"]
            parameters = []
            for row in group.itertuples(index=False):
                expressions.append(
                    f"sum(CASE WHEN {quoted} >= ? THEN 1 ELSE 0 END)"
                )
                parameters.append(float(row.threshold))
            values = connection.execute(
                f"SELECT {', '.join(expressions)} FROM read_parquet('{source}')",
                parameters,
            ).fetchone()
            valid_scores = int(values[0])
            previous_threshold = None
            for row, exceedances in zip(group.itertuples(index=False), values[1:]):
                rows.append({
                    "model_id": channel,
                    "threshold_quantile": float(row.threshold_quantile),
                    "threshold": float(row.threshold),
                    "blocks_used": int(row.blocks_used),
                    "unique_block_maxima": int(row.unique_block_maxima),
                    "expected_tail_blocks": float(row.expected_tail_blocks),
                    "threshold_tied_to_previous": (
                        previous_threshold is not None
                        and float(row.threshold) == previous_threshold
                    ),
                    "valid_scores": valid_scores,
                    "raw_exceedances": int(exceedances or 0),
                    "raw_exceedance_fraction": (
                        float(exceedances or 0) / valid_scores
                        if valid_scores else float("nan")
                    ),
                })
                previous_threshold = float(row.threshold)
    audit = pd.DataFrame(rows)
    for channel, group in audit.groupby("model_id"):
        ordered = group.sort_values("threshold_quantile")
        if ordered["threshold"].diff().dropna().lt(-1e-12).any():
            raise AssertionError(f"Thresholds decrease for {channel}")
        if ordered["raw_exceedances"].diff().dropna().gt(0).any():
            raise AssertionError(f"Exceedances increase for {channel}")
    return audit


score_coverage = score_missingness(score_path, available)
threshold_audit = threshold_exceedance_audit(
    threshold_score_path, thresholds
)
display(threshold_audit)
print(f"Threshold-estimation source: {threshold_score_path.name}")
print(f"Independent workload exposure: {verification_exposure:,.1f} entity-days")
print(f"Development exposure: {exposure:,.1f} entity-days")

## 4. Compare at a matched incident workload


In [ ]:
def metric_row(result, name):
    row = result["metrics"].loc[result["metrics"]["metric"].eq(name)]
    if row.empty:
        raise KeyError(f"Evaluation did not return {name!r}")
    return row.iloc[0]


def threshold_map(channels, quantile):
    return {
        channel: float(thresholds.loc[
            thresholds["model_id"].eq(channel)
            & thresholds["threshold_quantile"].eq(quantile),
            "threshold",
        ].iloc[0])
        for channel in channels
    }


def cases_from_grid(grid, channels, quantile):
    alerts = pd.concat(
        [grid[(channel, float(quantile))] for channel in channels],
        ignore_index=True,
    )
    if len(alerts):
        alerts = alerts.sort_values("alert_start").reset_index(drop=True)
        alerts["alert_id"] = [
            f"A-{number:09d}" for number in range(1, len(alerts) + 1)
        ]
    cases, members = form_cases(
        alerts,
        topology,
        gap_seconds=POLICY["incidents"]["quiet_period_seconds"],
        thresholds=threshold_map(channels, quantile),
        shared_scope_models=("group_common_mode",),
    )
    return alerts, cases, members


quantiles = sorted(thresholds["threshold_quantile"].unique())
budget = float(POLICY["workload"]["false_incidents_per_entity_day"])
budget_gate = budget * float(POLICY["workload"]["safety_factor"])
confidence = float(POLICY["workload"]["confidence_level"])
minimum_tail_blocks = float(
    POLICY["thresholds"]["minimum_expected_tail_blocks"]
)
verification_workload = {}
label_free_quantile = {}

for portfolio, channels in portfolios.items():
    admissible = []
    for quantile in quantiles:
        quantile = float(quantile)
        support = thresholds.loc[
            thresholds["model_id"].isin(channels),
            ["model_id", "threshold_quantile", "expected_tail_blocks"],
        ]
        support = support.loc[support["threshold_quantile"].eq(quantile)]
        support_ok = (
            len(support) == len(channels)
            and support["expected_tail_blocks"].ge(minimum_tail_blocks).all()
        )
        _, cases, _ = cases_from_grid(
            verification_grid, channels, quantile
        )
        rate = len(cases) / verification_exposure
        _, upper = poisson_rate_interval(
            len(cases), verification_exposure, confidence
        )
        verification_workload[(portfolio, quantile)] = {
            "incidents": len(cases),
            "rate": rate,
            "ci_high": upper,
            "threshold_support_ok": support_ok,
            "minimum_expected_tail_blocks": (
                float(support["expected_tail_blocks"].min())
                if len(support) else 0.0
            ),
        }
        if support_ok and upper <= budget_gate:
            admissible.append(quantile)
    # Ascending quantiles run from most to least sensitive.
    label_free_quantile[portfolio] = min(admissible) if admissible else None

rows = []
for portfolio, channels in portfolios.items():
    for quantile in quantiles:
        quantile = float(quantile)
        alerts, cases, members = cases_from_grid(
            alert_grid, channels, quantile
        )
        result = evaluate_cases(
            cases,
            members,
            events,
            intervals,
            exposure_value=exposure,
            exposure_unit="entity_day",
            decision_horizon_seconds=POLICY["evaluation"]["default_decision_horizon_seconds"],
            topology_memberships=topology,
        )
        recall = metric_row(result, "event_recall")
        precision = metric_row(result, "case_precision")
        false_rate = metric_row(result, "false_cases_per_entity_day")
        durations = (
            pd.to_datetime(alerts["alert_end"], utc=True)
            - pd.to_datetime(alerts["alert_start"], utc=True)
        ).dt.total_seconds() / 3600 if len(alerts) else pd.Series(dtype=float)
        workload = verification_workload[(portfolio, quantile)]
        rows.append({
            "candidate_key": f"{portfolio}|q={quantile:.6g}",
            "portfolio": portfolio,
            "candidate": portfolio,
            "channels": ", ".join(channels),
            "threshold_quantile": quantile,
            "selection_eligible": selection_eligible[portfolio],
            "label_free_choice": label_free_quantile[portfolio] == quantile,
            "verification_incidents": workload["incidents"],
            "verification_incidents_per_entity_day": workload["rate"],
            "verification_rate_ci_high": workload["ci_high"],
            "threshold_support_ok": workload["threshold_support_ok"],
            "minimum_expected_tail_blocks": workload["minimum_expected_tail_blocks"],
            "raw_alerts": len(alerts),
            "median_alert_hours": durations.median() if len(durations) else 0.0,
            "p95_alert_hours": durations.quantile(0.95) if len(durations) else 0.0,
            "incidents": len(cases),
            "scoreable_faults": int(recall["denominator"]),
            "event_recall": float(recall["value"]),
            "event_recall_ci_low": float(recall["ci_low"]),
            "event_recall_ci_high": float(recall["ci_high"]),
            "incident_precision": float(precision["value"]),
            "false_incidents_per_entity_day": float(false_rate["value"]),
            "false_rate_ci_high": float(false_rate["ci_high"]),
            "false_incidents_per_entity_day_ci_high": float(false_rate["ci_high"]),
            "median_detection_delay_seconds": float(
                metric_row(result, "median_detection_delay_seconds")["value"]
            ),
            "maximum_missing_score_fraction": max(
                score_coverage[channel]["entity_p95"] for channel in channels
            ),
            "missing_score_fraction": max(
                score_coverage[channel]["entity_p95"] for channel in channels
            ),
        })

comparison = pd.DataFrame(rows)
display(comparison.sort_values(
    ["false_incidents_per_entity_day", "event_recall"],
    ascending=[True, False],
))
print("Label-free choices from independent late-calibration workload:")
display(comparison.loc[comparison["label_free_choice"], [
    "portfolio", "selection_eligible", "threshold_quantile",
    "threshold_support_ok", "minimum_expected_tail_blocks",
    "verification_incidents_per_entity_day", "verification_rate_ci_high",
    "event_recall", "event_recall_ci_low",
    "false_incidents_per_entity_day_ci_high",
]])

## 5. Apply the fail-closed gate


In [ ]:
gate = POLICY["selection"]
preference = list(gate["eligible_portfolios"])
admissible_comparison = comparison.loc[
    comparison["label_free_choice"] & comparison["selection_eligible"]
].copy()
selected, decision = select_development_candidate(
    admissible_comparison,
    development_faults=int(events["fault_id"].nunique()),
    false_incident_budget=POLICY["workload"]["false_incidents_per_entity_day"],
    budget_safety_factor=POLICY["workload"]["safety_factor"],
    minimum_faults=gate["minimum_development_faults"],
    minimum_recall=gate["minimum_development_event_recall"],
    maximum_missing_score_fraction=gate["maximum_missing_score_fraction"],
    portfolio_preference=preference,
)

diagnostic_pool = admissible_comparison
if diagnostic_pool.empty:
    diagnostic_pool = comparison.loc[comparison["selection_eligible"]]
diagnostic = diagnostic_pool.sort_values(
    ["event_recall", "false_incidents_per_entity_day_ci_high"],
    ascending=[False, True],
).iloc[0]


def configuration(row, status, selection_basis):
    channels = row["channels"].split(", ")
    quantile = float(row["threshold_quantile"])

    def number(name):
        value = row[name]
        if pd.isna(value):
            return None
        return int(value) if name in {"scoreable_faults", "verification_incidents"} else float(value)

    return {
        "status": status,
        "selection_basis": selection_basis,
        "candidate": str(row["candidate"]),
        "channels": channels,
        "threshold_quantile": quantile,
        "threshold_support": {
            "minimum_required_tail_blocks": minimum_tail_blocks,
            "minimum_observed_tail_blocks": number(
                "minimum_expected_tail_blocks"
            ),
            "passed": bool(row["threshold_support_ok"]),
        },
        "thresholds": {
            channel: float(thresholds.loc[
                thresholds["model_id"].eq(channel)
                & thresholds["threshold_quantile"].eq(quantile), "threshold"
            ].iloc[0])
            for channel in channels
        },
        "persistence_observations": {
            channel: int(persistence[channel]) for channel in channels
        },
        "recovery_observations": int(RECOVERY_OBSERVATIONS),
        "recovery_threshold_fraction": RECOVERY_THRESHOLD_FRACTION,
        "cadence_seconds": CADENCE_SECONDS,
        "incident_quiet_period_seconds": POLICY["incidents"]["quiet_period_seconds"],
        "verification_workload": {
            name: number(name)
            for name in (
                "verification_incidents",
                "verification_incidents_per_entity_day",
                "verification_rate_ci_high",
            )
        },
        "development_metrics": {
            name: number(name)
            for name in (
                "scoreable_faults", "event_recall", "event_recall_ci_low",
                "event_recall_ci_high", "incident_precision",
                "false_incidents_per_entity_day", "false_rate_ci_high",
                "median_detection_delay_seconds", "maximum_missing_score_fraction",
            )
        },
        "model_run_id": MODEL_RUN_ID,
        "model_manifest_sha256": file_sha256(MODEL_ROOT / "model_manifest.json"),
        "resolved_policy_sha256": model_manifest["resolved_policy_sha256"],
        "selection_run_id": SELECTION_RUN_ID,
        "development_truth_manifest_sha256": file_sha256(
            TRUTH_ROOT / "truth_manifest.json"
        ),
        "evaluation_module_sha256": EVALUATION_MODULE_SHA256,
        "selection_module_sha256": SELECTION_MODULE_SHA256,
        "holdout_opened": False,
    }


label_free_rows = comparison.loc[comparison["label_free_choice"]].copy()
label_free_configurations = [
    configuration(
        row,
        "LABEL_FREE_CALIBRATION",
        "independent_late_calibration_workload",
    )
    for _, row in label_free_rows.iterrows()
]

selection_status = {
    "model_manifest_sha256": file_sha256(MODEL_ROOT / "model_manifest.json"),
    "resolved_policy_sha256": model_manifest["resolved_policy_sha256"],
    "development_truth_manifest_sha256": file_sha256(
        TRUTH_ROOT / "truth_manifest.json"
    ),
    "evaluation_module_sha256": EVALUATION_MODULE_SHA256,
    "selection_module_sha256": SELECTION_MODULE_SHA256,
    **decision,
    "result": "PASS" if selected is not None else "STOP",
    "reason": (
        "At least one development candidate met every frozen gate."
        if selected is not None
        else "No development candidate met every frozen gate; holdout remains sealed."
    ),
    "holdout_opened": False,
    "label_free_configurations": len(label_free_configurations),
    "minimum_expected_tail_blocks": minimum_tail_blocks,
    "workload_verification_independent_of_threshold_fit": True,
}
display(pd.Series(selection_status, name="result").to_frame())

## 6. Publish only compact development evidence


In [ ]:
if OUTPUT_ROOT.exists():
    previous = read_json(OUTPUT_ROOT / "selection_status.json")
    require_same(
        previous,
        model_manifest_sha256=selection_status["model_manifest_sha256"],
        resolved_policy_sha256=selection_status["resolved_policy_sha256"],
        development_truth_manifest_sha256=(
            selection_status["development_truth_manifest_sha256"]
        ),
        evaluation_module_sha256=selection_status["evaluation_module_sha256"],
        selection_module_sha256=selection_status["selection_module_sha256"],
    )
    print("Using existing immutable selection:", previous["result"])
else:
    with immutable_output_directory(OUTPUT_ROOT) as output:
        comparison.to_parquet(output / "development_comparison.parquet", index=False)
        write_json(output / "selection_status.json", selection_status)
        write_json(
            output / "label_free_configurations.json",
            label_free_configurations,
        )
        write_json(output / "best_diagnostic_configuration.json", configuration(
            diagnostic, "DIAGNOSTIC_ONLY_NOT_DEPLOYABLE", "label_free_threshold_then_development_diagnosis"
        ))
        if selected is not None:
            write_json(output / "selected_configuration.json", configuration(
                selected, "DEVELOPMENT_GATES_PASSED", "development_labels"
            ))

assert not selection_status["holdout_opened"]
if selected is None:
    print("STOP — no deployable configuration was written")
    print("Use the diagnostic result for diagnosis only; do not open holdout truth")
else:
    print("PASS — one frozen configuration is ready for incident formation")
print("Next: 08_ALERTS_INCIDENTS_AND_DYING_GASP.ipynb")
